In [42]:
from dotenv import load_dotenv
load_dotenv()

import base64
from scrapling.fetchers import FetcherSession
from aiofiles import open
from curl_cffi import AsyncSession
from typing import cast
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from enum import StrEnum

## Crawling

In [3]:
with FetcherSession(impersonate='chrome') as session:
    page = session.get(
        'https://doe.gov.ph/articles/group/liquid-fuels?maincat=Retail%20Pump%20Prices&subcategory=NCR%20Pump%20Prices&display_type=Card'
    )

[2026-04-28 19:30:22] INFO: Fetched (200) <GET https://doe.gov.ph/articles/group/liquid-fuels?maincat=Retail%20Pump%20Prices&subcategory=NCR%20Pump%20Prices&display_type=Card> (referer: https://www.google.com/)


In [17]:
link_elements = (
    page
    .css('div.ex1')
    .css('a[href^="https://prod-cms.doe.gov.ph"]')
)
links = [l.attrib.get('href') for l in link_elements]
links

['https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-03312026-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-04072026-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-04142026-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-04212026-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-02242026-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-03032026-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-03102026n-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-03172026-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-03242026-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-02032026-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-02102026-1-pdf',
 'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-0217

In [30]:
async with AsyncSession() as session:
    session = cast(AsyncSession, session)
    filename = f"{links[0].split('/')[-1]}.pdf"
    res = cast(
        AsyncSession,
        await session.get(links[0], stream=True, impersonate='chrome')
    )
    async with open(f'./data/pdfs/{filename}', 'wb') as f:
        async for chunk in res.aiter_content(65536):
            await f.write(chunk)

## OCR

In [67]:
links[0]

'https://prod-cms.doe.gov.ph/documents/d/guest/ncr-price-monitoring-03312026-pdf'

In [68]:
class FuelType(StrEnum):
    RON100 = 'RON100'
    RON97 = 'RON97'
    RON95 = 'RON95'
    RON91 = 'RON91'
    DIESEL = 'DIESEL'
    DIESEL_PLUS = 'DIESEL_PLUS'
    KEROSENE = 'KEROSENE'
    
class FuelBrand(StrEnum):
    PETRON = 'PETRON'
    SHELL = 'SHELL'
    CALTEX = 'CALTEX'
    PHOENIX = 'PHOENIX'
    TOTAL = 'TOTAL'
    FLYING_V = 'FLYING_V'
    UNIOIL = 'UNIOIL'
    SEAOIL = 'SEAOIL'
    PTT = 'PTT'
    INDEPENDENT = 'INDEPENDENT'

class FuelPrice(BaseModel):
    product: FuelType
    min_price: float | None = Field(None, ge=0.0, description="Minimum of the price range")
    max_price: float | None = Field(None, ge=0.0, description="Maximum of the price range")
    common_price: float | None = Field(None, ge=0.0)
    brand: FuelBrand

class FuelPriceCity(BaseModel):
    area: str = Field(description="Name of the area/city")
    fuel_prices: list[FuelPrice]
    
class StructuredOutput(BaseModel):
    results: list[FuelPriceCity]

model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    timeout=60,
    max_retries=3,
)
agent = create_agent(
    model,
    system_prompt="""
    You are an expert OCR Agent in the energy industry. Your role is to read the 
    provided PDF(s) containing fuel pump prices and return a structured output.
    """,
    response_format=StructuredOutput,
)

In [71]:
async with open(f'./data/pdfs/{filename}', 'rb') as f:
    file_data = base64.b64encode(
        await f.read()
    ).decode()

res = await agent.ainvoke({
    'messages': [
        HumanMessage(content=[
            {
                'type': 'media',
                'mime_type': 'application/pdf',
                'data': file_data,
            },
        ])
    ]
})
res

{'messages': [HumanMessage(content=[{'type': 'media', 'mime_type': 'application/pdf', 'data': '...'}], additional_kwargs={}, response_metadata={}, id='f287a4ee-3334-4535-9c6d-44fe800d203c'),
  AIMessage(content=[{'type': 'text', 'text': '{\n  "results": [\n    {\n      "area": "Caloocan City",\n      "fuel_prices": [\n        {\n          "product": "RON97",\n          "brand": "CALTEX"\n        },\n        {\n          "product": "RON95",\n          "brand": "PETRON"\n        },\n        {\n          "product": "RON95",\n          "brand": "SHELL"\n        },\n        {\n          "product": "RON95",\n          "brand": "CALTEX"\n        },\n        {\n          "product": "RON95",\n          "brand": "FLYING_V"\n        },\n        {\n          "product": "RON95",\n          "brand": "UNIOIL"\n        },\n        {\n          "product": "RON95",\n          "brand": "SEAOIL"\n        },\n        {\n          "product": "RON95",\n          "brand": "INDEPENDENT"\n        },\n        {\

In [73]:
res['structured_response'].model_dump()

{'results': [{'area': 'Caloocan City',
   'fuel_prices': [{'product': <FuelType.RON97: 'RON97'>,
     'min_price': None,
     'max_price': None,
     'common_price': None,
     'brand': <FuelBrand.CALTEX: 'CALTEX'>},
    {'product': <FuelType.RON95: 'RON95'>,
     'min_price': None,
     'max_price': None,
     'common_price': None,
     'brand': <FuelBrand.PETRON: 'PETRON'>},
    {'product': <FuelType.RON95: 'RON95'>,
     'min_price': None,
     'max_price': None,
     'common_price': None,
     'brand': <FuelBrand.SHELL: 'SHELL'>},
    {'product': <FuelType.RON95: 'RON95'>,
     'min_price': None,
     'max_price': None,
     'common_price': None,
     'brand': <FuelBrand.CALTEX: 'CALTEX'>},
    {'product': <FuelType.RON95: 'RON95'>,
     'min_price': None,
     'max_price': None,
     'common_price': None,
     'brand': <FuelBrand.FLYING_V: 'FLYING_V'>},
    {'product': <FuelType.RON95: 'RON95'>,
     'min_price': None,
     'max_price': None,
     'common_price': None,
     'brand